# Bilişsel Performans Tahmini - Şampiyon Model (CatBoost Native)

**Hedef:** Karmaşık ve gürültülü modelleme tekniklerinden arındırılmış, Occam'ın Usturası prensibiyle sadece doğruluğu kanıtlanmış adımları içeren nihai ve temiz çalışma alanı.

## 1. Veri Ön İşleme (Data Preprocessing)

Bu aşamada şu katı kurallar uygulanmıştır:
* **Leakage (Sızıntı) Koruması:** Eksik sayısal verileri doldurmak için kullanılacak `median` (medyan) değerleri, geleceği görmemek adına *sadece* eğitim setinden (train) hesaplanmış ve test setine öyle aktarılmıştır.
* **Ham Kategorik Veri:** CatBoost'un dahili algoritmasını bozmamak için hiçbir One-Hot Encoding (OHE) işlemi yapılmamış, tüm kategorik değişkenler `string` (metin) formatında bırakılmıştır.
* **Altın Sinyaller:** Onlarca deneme arasından Local RMSE skorumuzu `1.2178` seviyesine indiren yegane 2 özellik (`bilissel_yuk_endeksi` ve `uyku_hijyeni_ihlali`) sisteme entegre edilmiştir.

In [4]:
import pandas as pd
import numpy as np

print("Tertemiz bir sayfa: Veri Ön İşleme (Preprocessing) başlıyor...")

# 1. Verileri Okuma
train = pd.read_csv('train.csv')
test = pd.read_csv('test_x.csv')

# 2. Sütun Kategorizasyonları
kategorik_sutunlar = ['cinsiyet', 'meslek', 'ulke', 'kronotip', 'ruh_sagligi_durumu', 'mevsim', 'gun_tipi']
kapatilacak_sutunlar = ['id', 'bilissel_performans_skoru']
numerik_sutunlar = [col for col in train.columns if col not in kategorik_sutunlar + kapatilacak_sutunlar]

# 3. Eksik Veri Yönetimi (Strictly No Leakage)
for col in numerik_sutunlar:
    # Medyan sadece Train setinden öğrenilir
    medyan_deger = train[col].median()
    train[col].fillna(medyan_deger, inplace=True)
    test[col].fillna(medyan_deger, inplace=True)

for col in kategorik_sutunlar:
    train[col].fillna('Bilinmiyor', inplace=True)
    train[col] = train[col].astype(str)
    
    test[col].fillna('Bilinmiyor', inplace=True)
    test[col] = test[col].astype(str)

# 4. Altın Özellik Mühendisliği (Sadece kanıtlanmış 2 sinyal)
for df in [train, test]:
    df['bilissel_yuk_endeksi'] = df['stres_skoru'] * (df['gunluk_calisma_saati'] / (df['derin_uyku_yuzdesi'] + 0.1))
    df['uyku_hijyeni_ihlali'] = (df['uyku_oncesi_kafein_mg'] > 50).astype(int) + (df['uyku_oncesi_ekran_suresi_dk'] > 60).astype(int)

# 5. Model İçin Ayırma İşlemleri (X ve y)
X = train.drop(columns=['id', 'bilissel_performans_skoru'])
y = train['bilissel_performans_skoru']
X_test = test.drop(columns=['id']) # Final Kaggle submission için hazır

print("-" * 50)
print("Veri Ön İşleme Tamamlandı. Matris Eğitime Hazır!")
print(f"Eğitim Matrisi (X) Boyutu : {X.shape}")
print(f"Test Matrisi (X_test) Boyutu: {X_test.shape}")

Tertemiz bir sayfa: Veri Ön İşleme (Preprocessing) başlıyor...
--------------------------------------------------
Veri Ön İşleme Tamamlandı. Matris Eğitime Hazır!
Eğitim Matrisi (X) Boyutu : (56000, 24)
Test Matrisi (X_test) Boyutu: (24000, 24)


/var/folders/s0/tf_n875s3ps06ts4061yxy7c0000gn/T/ipykernel_11748/1912606636.py:19: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train[col].fillna(medyan_deger, inplace=True)
/var/folders/s0/tf_n875s3ps06ts4061yxy7c0000gn/T/ipykernel_11748/1912606636.py:20: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values alway

## 2. Model Eğitimi ve Validasyon (CatBoost K-Fold)

**Strateji:** Modelin ezberleme (overfitting) yapmasını engellemek ve Kaggle'ın gizli test setindeki performansını en gerçekçi şekilde simüle etmek için 5 Katlı Çapraz Doğrulama (5-Fold Cross Validation) kullanılmaktadır. 

Model, daha önce Optuna ile belirlenen ve veri setine en uygun olduğu kanıtlanan sabit hiperparametrelerle eğitilecektir. Amacımız, tertemiz matrisimizle referans skorumuz olan `1.2178`'i (veya daha iyisini) tekrar teyit etmektir.

In [5]:
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import numpy as np

# K-Fold Cross Validation ayarları
kf = KFold(n_splits=5, shuffle=True, random_state=42)
rmse_skorlari = []

print("Şampiyon CatBoost Modeli (Native) Eğitiliyor...")
print("-" * 50)

for fold, (train_index, val_index) in enumerate(kf.split(X)):
    X_tr, X_val = X.iloc[train_index], X.iloc[val_index]
    y_tr, y_val = y.iloc[train_index], y.iloc[val_index]
    
    # Doğruluğu kanıtlanmış nihai parametrelerimiz
    model = CatBoostRegressor(
        learning_rate=0.045364, 
        depth=5, 
        l2_leaf_reg=5.7870,
        iterations=1000, 
        bootstrap_type='Bayesian',
        cat_features=kategorik_sutunlar, 
        random_state=42, 
        verbose=0  # Eğitimi sessiz modda çalıştırır
    )
    
    # Modeli eğitiyoruz
    model.fit(X_tr, y_tr)
    
    # Validasyon seti üzerinde tahmin yapıyoruz
    preds = model.predict(X_val)
    
    # Bu fold için hatayı hesaplıyoruz
    fold_rmse = np.sqrt(mean_squared_error(y_val, preds))
    rmse_skorlari.append(fold_rmse)
    print(f"Fold {fold+1} RMSE Skoru: {fold_rmse:.4f}")

# Nihai ortalama skoru hesaplıyoruz
final_skor = np.mean(rmse_skorlari)

print("-" * 50)
print(f"NİHAİ LOCAL RMSE SKORU : {final_skor:.4f}")

if final_skor <= 1.2179:
    print("TEBRİKLER! Matris temizlendi, sızıntılar kapatıldı ve şampiyon skor sapağlam duruyor.")
else:
    print("DİKKAT: Skorda kayma var, temizlik sırasında bir sinyal etkilenmiş olabilir.")

Şampiyon CatBoost Modeli (Native) Eğitiliyor...
--------------------------------------------------
Fold 1 RMSE Skoru: 1.2217
Fold 2 RMSE Skoru: 1.2173
Fold 3 RMSE Skoru: 1.2046
Fold 4 RMSE Skoru: 1.2122
Fold 5 RMSE Skoru: 1.2333
--------------------------------------------------
NİHAİ LOCAL RMSE SKORU : 1.2178
TEBRİKLER! Matris temizlendi, sızıntılar kapatıldı ve şampiyon skor sapağlam duruyor.


## 2 Aşama: Büyük Final - Tohum Harmanlama (Seed Blending)

**Amaç:** K-Fold sırasında farklı alt kümelerde oluşan dalgalanmaları (Örn: Fold 3'teki 1.20 ile Fold 5'teki 1.23 arası uçurum) matematiksel olarak pürüzsüzleştirmek.

**Metodoloji:**
Ağaçların bölünme noktalarını belirlerken kullandığı rastgelelik tohumu (random seed) değiştirilerek aynı CatBoost modeli 5 farklı "paralel evrende" yeniden eğitilir. Her evrenin hataları farklı noktalarda birikeceği için, bu 5 evrenin tahminlerinin ortalaması alındığında hatalar birbirini nötralize (sönümleme) eder ve modelin genel varyansı düşer.

In [8]:
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

# 1. Beş Farklı 'Evren' (Random Seed) Belirliyoruz
seeds = [42, 1907, 2026, 777, 99]

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

# K-Fold yapısını SABİT tutuyoruz ki veri bölünmesi değil, sadece modelin iç rastgeleliği değişsin
kf = KFold(n_splits=5, shuffle=True, random_state=42) 

print("Şampiyonlar Ligi Başlıyor: Tohum Harmanlama (Seed Blending) Devrede...")
print("-" * 50)

for seed in seeds:
    seed_oof = np.zeros(len(X))
    seed_test = np.zeros(len(X_test))
    
    for train_idx, val_idx in kf.split(X):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        # Tek değişen parametre: random_state=seed
        model = CatBoostRegressor(
            learning_rate=0.045364, 
            depth=5, 
            l2_leaf_reg=5.7870,
            iterations=1000, 
            bootstrap_type='Bayesian',
            cat_features=kategorik_sutunlar, 
            random_state=seed, 
            verbose=0
        )
        model.fit(X_tr, y_tr)
        
        # Doğrulama (OOF) tahminlerini alıyoruz
        seed_oof[val_idx] = model.predict(X_val)
        
        # Test seti için her fold'un tahminini toplayıp 5'e bölüyoruz
        seed_test += model.predict(X_test) / 5 
        
    # Bu tohumun kendi RMSE'sini hesaplayıp ekrana yazdıralım
    seed_rmse = np.sqrt(mean_squared_error(y, seed_oof))
    print(f"Seed {seed} Evreni Local RMSE: {seed_rmse:.4f}")
    
    # Ana harmanlama (blend) havuzuna bu tohumun sonuçlarını eşit ağırlıkla ekliyoruz
    oof_preds += seed_oof / len(seeds)
    test_preds += seed_test / len(seeds)

# 2. Harmanlanmış Nihai Skoru Hesaplama
final_blend_rmse = np.sqrt(mean_squared_error(y, oof_preds))

print("-" * 50)
print(f"MUTLAK ZAFER! NİHAİ HARMANLANMIŞ (BLENDED) LOCAL RMSE: {final_blend_rmse:.6f}")

# 3. Kaggle Submission Dosyasını Oluşturma
submission = pd.DataFrame({
    'id': test['id'], 
    'bilissel_performans_skoru': test_preds
})

dosya_adi = 'SAMPİYON_SEED_BLEND_FINAL.csv'
submission.to_csv(dosya_adi, index=False)

print(f"Kaggle Gönderim Dosyası Hazır: {dosya_adi}")

Şampiyonlar Ligi Başlıyor: Tohum Harmanlama (Seed Blending) Devrede...
--------------------------------------------------
Seed 42 Evreni Local RMSE: 1.2179
Seed 1907 Evreni Local RMSE: 1.2186
Seed 2026 Evreni Local RMSE: 1.2187
Seed 777 Evreni Local RMSE: 1.2179
Seed 99 Evreni Local RMSE: 1.2184
--------------------------------------------------
MUTLAK ZAFER! NİHAİ HARMANLANMIŞ (BLENDED) LOCAL RMSE: 1.217160
Kaggle Gönderim Dosyası Hazır: SAMPİYON_SEED_BLEND_FINAL.csv


In [9]:
import pandas as pd
import numpy as np

train = pd.read_csv('train.csv')
y = train['bilissel_performans_skoru']

print("KAGGLE 'MAGIC' DEDEKTÖRÜ ÇALIŞIYOR...")
print("-" * 50)

# 1. Tam Sayı (Discrete) Kontrolü
ondalikli_sayi_orani = (y % 1 != 0).mean()
if ondalikli_sayi_orani == 0:
    print("🚨 BÜYÜK İPUCU: Hedef değişken %100 tam sayılardan oluşuyor! (Yuvarlama hilesi işe yarar)")
else:
    print(f"Hedef değişkende ondalıklı sayılar var. (Oran: %{ondalikli_sayi_orani*100:.1f})")

# 2. Alt ve Üst Sınır (Clipping) Kontrolü
print(f"Skor Sınırları -> Minimum: {y.min():.2f} | Maksimum: {y.max():.2f}")
print("Tahminlerimizi bu sınırlara 'clip'lemek hayat kurtarabilir.")

# 3. Kategori Bazlı Farklılıklar (Segmentasyon İhtiyacı)
print("-" * 50)
print("Bazı kategorilerde dağılımlar aşırı uçuk mu?")
for col in ['kronotip', 'gun_tipi', 'ruh_sagligi_durumu']:
    if col in train.columns:
        grup_farki = train.groupby(col)['bilissel_performans_skoru'].agg(['min', 'max', 'mean', 'std']).round(2)
        print(f"\n[{col}] Sütunu Dağılımı:\n{grup_farki}")

KAGGLE 'MAGIC' DEDEKTÖRÜ ÇALIŞIYOR...
--------------------------------------------------
Hedef değişkende ondalıklı sayılar var. (Oran: %98.2)
Skor Sınırları -> Minimum: 0.00 | Maksimum: 10.00
Tahminlerimizi bu sınırlara 'clip'lemek hayat kurtarabilir.
--------------------------------------------------
Bazı kategorilerde dağılımlar aşırı uçuk mu?

[kronotip] Sütunu Dağılımı:
              min   max  mean   std
kronotip                           
Gece insani   0.0  10.0  5.80  2.24
Notr          0.0  10.0  5.93  2.23
Sabah insani  0.0  10.0  6.00  2.23

[gun_tipi] Sütunu Dağılımı:
            min   max  mean   std
gun_tipi                         
Hafta ici   0.0  10.0  5.48  2.20
Hafta sonu  0.0  10.0  6.99  1.94

[ruh_sagligi_durumu] Sütunu Dağılımı:
                        min    max  mean   std
ruh_sagligi_durumu                            
Anksiyete               0.0  10.00  5.01  2.10
Anksiyete ve depresyon  0.0   9.39  3.59  2.05
Depresyon               0.0   9.67  4.39  2.01
Sag

## 16. Aşama: Büyük Final - Kaggle Magic (Sosyolojik Etkileşim + Clipping)

**Uygulanan Sihirler (Magic):**
1.  **Sosyolojik Uçurum Sinyalleri:** Veri analizinde tespit edilen en düşük ortalamaya sahip grup (Hafta içi + Anksiyete/Depresyon) ile en yüksek ortalamaya sahip grup (Hafta sonu + Sağlıklı) için modele net birer "Kısa Yol" (Interaction Feature) açıldı.
2.  **Sınır Tıraşlaması (Clipping):** Hedef değişkenin kesin sınırları olan `[0, 10]` aralığı, modelin ürettiği tüm tahminlere zorunlu kılındı. Böylece RMSE formülündeki o ölümcül cezalar (penalty) tamamen ortadan kaldırıldı.

In [10]:
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

print("Kaggle Magic Devrede: Sosyolojik Sinyaller ve Sınır Tıraşlaması (Clipping)...")
print("-" * 50)

# 1. Tertemiz Veriyi Okuma
train = pd.read_csv('train.csv')
test = pd.read_csv('test_x.csv')

kategorik_sutunlar = ['cinsiyet', 'meslek', 'ulke', 'kronotip', 'ruh_sagligi_durumu', 'mevsim', 'gun_tipi']
numerik_sutunlar = [c for c in train.columns if c not in kategorik_sutunlar + ['id', 'bilissel_performans_skoru']]

# 2. Sızıntısız Ön İşleme
for col in numerik_sutunlar:
    med = train[col].median()
    train[col].fillna(med, inplace=True)
    test[col].fillna(med, inplace=True)

for col in kategorik_sutunlar:
    train[col].fillna('Bilinmiyor', inplace=True)
    test[col].fillna('Bilinmiyor', inplace=True)
    train[col] = train[col].astype(str)
    test[col] = test[col].astype(str)

# 3. ÖZELLİK MÜHENDİSLİĞİ (Altın Sinyaller + Kaggle Magic)
for df in [train, test]:
    # Eski Altın Sinyaller
    df['bilissel_yuk_endeksi'] = df['stres_skoru'] * (df['gunluk_calisma_saati'] / (df['derin_uyku_yuzdesi'] + 0.1))
    df['uyku_hijyeni_ihlali'] = (df['uyku_oncesi_kafein_mg'] > 50).astype(int) + (df['uyku_oncesi_ekran_suresi_dk'] > 60).astype(int)
    
    # YENİ: Sosyolojik Uçurum Sinyalleri (Magic)
    df['kriz_durumu'] = ((df['gun_tipi'] == 'Hafta ici') & (df['ruh_sagligi_durumu'] == 'Anksiyete ve depresyon')).astype(int)
    df['zirve_durumu'] = ((df['gun_tipi'] == 'Hafta sonu') & (df['ruh_sagligi_durumu'] == 'Saglikli')).astype(int)

# Matrisleri Ayırma
X = train.drop(columns=['id', 'bilissel_performans_skoru'])
y = train['bilissel_performans_skoru']
X_test = test.drop(columns=['id'])

# 4. TOHUM HARMANLAMA (Seed Blending) EĞİTİMİ
seeds = [42, 1907, 2026, 777, 99]
oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for seed in seeds:
    seed_oof = np.zeros(len(X))
    seed_test = np.zeros(len(X_test))
    
    for train_idx, val_idx in kf.split(X):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model = CatBoostRegressor(
            learning_rate=0.045364, depth=5, l2_leaf_reg=5.7870,
            iterations=1000, bootstrap_type='Bayesian',
            cat_features=kategorik_sutunlar, 
            random_state=seed, verbose=0
        )
        model.fit(X_tr, y_tr)
        
        seed_oof[val_idx] = model.predict(X_val)
        seed_test += model.predict(X_test) / 5
        
    oof_preds += seed_oof / len(seeds)
    test_preds += seed_test / len(seeds)

# 5. YENİ: SINIR TIRAŞLAMASI (CLIPPING - MAGIC)
# Tahminleri 0 ile 10 arasına zorluyoruz
oof_preds_clipped = np.clip(oof_preds, 0.0, 10.0)
test_preds_clipped = np.clip(test_preds, 0.0, 10.0)

# Nihai Skoru Hesaplama
final_blend_rmse = np.sqrt(mean_squared_error(y, oof_preds_clipped))

print(f"HARMANLANMIŞ VE TIRAŞLANMIŞ (MAGIC) LOCAL RMSE: {final_blend_rmse:.6f}")
print("-" * 50)

# Kaggle Submission Dosyasını Oluşturma
submission = pd.DataFrame({
    'id': test['id'], 
    'bilissel_performans_skoru': test_preds_clipped
})
submission.to_csv('KAGGLE_MAGIC_SUBMISSION.csv', index=False)
print("DOSYA HAZIR: KAGGLE_MAGIC_SUBMISSION.csv")

Kaggle Magic Devrede: Sosyolojik Sinyaller ve Sınır Tıraşlaması (Clipping)...
--------------------------------------------------


/var/folders/s0/tf_n875s3ps06ts4061yxy7c0000gn/T/ipykernel_11748/190632995.py:20: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train[col].fillna(med, inplace=True)
/var/folders/s0/tf_n875s3ps06ts4061yxy7c0000gn/T/ipykernel_11748/190632995.py:21: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves a

HARMANLANMIŞ VE TIRAŞLANMIŞ (MAGIC) LOCAL RMSE: 1.217518
--------------------------------------------------
DOSYA HAZIR: KAGGLE_MAGIC_SUBMISSION.csv


In [11]:
import optuna
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

print("Optuna Gece Mesaisi Başlıyor: Early Stopping Devrede...")

# 1. VERİ HAZIRLIĞI (Kaggle Magic Matrisi)
train = pd.read_csv('train.csv')
kategorik_sutunlar = ['cinsiyet', 'meslek', 'ulke', 'kronotip', 'ruh_sagligi_durumu', 'mevsim', 'gun_tipi']
numerik_sutunlar = [c for c in train.columns if c not in kategorik_sutunlar + ['id', 'bilissel_performans_skoru']]

for col in numerik_sutunlar:
    train[col].fillna(train[col].median(), inplace=True)
for col in kategorik_sutunlar:
    train[col].fillna('Bilinmiyor', inplace=True)
    train[col] = train[col].astype(str)

train['bilissel_yuk_endeksi'] = train['stres_skoru'] * (train['gunluk_calisma_saati'] / (train['derin_uyku_yuzdesi'] + 0.1))
train['uyku_hijyeni_ihlali'] = (train['uyku_oncesi_kafein_mg'] > 50).astype(int) + (train['uyku_oncesi_ekran_suresi_dk'] > 60).astype(int)
train['kriz_durumu'] = ((train['gun_tipi'] == 'Hafta ici') & (train['ruh_sagligi_durumu'] == 'Anksiyete ve depresyon')).astype(int)
train['zirve_durumu'] = ((train['gun_tipi'] == 'Hafta sonu') & (train['ruh_sagligi_durumu'] == 'Saglikli')).astype(int)

X = train.drop(columns=['id', 'bilissel_performans_skoru'])
y = train['bilissel_performans_skoru']

# 2. OPTUNA OBJECTIVE FONKSİYONU
def objective(trial):
    # Optuna'nın arayacağı parametre uzayı (iterations YOK!)
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 9),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        'random_strength': trial.suggest_float('random_strength', 0.1, 5.0),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'border_count': trial.suggest_int('border_count', 32, 255),
        
        # Claude'un tavsiyesi: İterasyon tavanı çok yüksek, kararı early_stopping verecek
        'iterations': 5000, 
        'eval_metric': 'RMSE',
        'bootstrap_type': 'Bayesian',
        'cat_features': kategorik_sutunlar,
        'random_state': 42,
        'verbose': 0
    }
    
    # Optimizasyon hızlansın diye 3-Fold kullanıyoruz
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    rmse_scores = []
    
    for train_idx, val_idx in kf.split(X):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model = CatBoostRegressor(**params)
        
        # Early stopping burada devreye giriyor! Hata 50 iterasyon boyunca düşmezse eğitimi keser.
        model.fit(X_tr, y_tr, 
                  eval_set=[(X_val, y_val)], 
                  early_stopping_rounds=50, 
                  verbose=0)
        
        preds = model.predict(X_val)
        preds_clipped = np.clip(preds, 0.0, 10.0) # Magic Clipping kuralımız
        rmse = np.sqrt(mean_squared_error(y_val, preds_clipped))
        rmse_scores.append(rmse)
        
    return np.mean(rmse_scores)

# 3. GECE BOYU ÇALIŞACAK MOTOR
# 100 deneme ortalama 1-2 saat sürecektir. Bilgisayarın gücüne göre n_trials'ı artırabilirsin.
study = optuna.create_study(direction='minimize')
print("Optuna hiperparametre uzayını taramaya başladı. Arkanıza yaslanın...")
study.optimize(objective, n_trials=100) 

print("-" * 50)
print("EN İYİ PARAMETRELER:")
for key, value in study.best_params.items():
    print(f"'{key}': {value},")
print(f"ULAŞILAN EN DÜŞÜK LOCAL RMSE: {study.best_value:.6f}")

[I 2026-05-10 23:05:15,713] A new study created in memory with name: no-name-070a8b15-320b-411a-af8e-2d6abfc59339


Optuna Gece Mesaisi Başlıyor: Early Stopping Devrede...
Optuna hiperparametre uzayını taramaya başladı. Arkanıza yaslanın...


[I 2026-05-10 23:07:06,554] Trial 0 finished with value: 1.2194420886556452 and parameters: {'learning_rate': 0.005930756481777138, 'depth': 7, 'l2_leaf_reg': 8.048591977665318, 'random_strength': 2.071661278087692, 'bagging_temperature': 0.5481867935117146, 'border_count': 168}. Best is trial 0 with value: 1.2194420886556452.
[I 2026-05-10 23:07:31,948] Trial 1 finished with value: 1.220094370319465 and parameters: {'learning_rate': 0.03603704413458784, 'depth': 6, 'l2_leaf_reg': 7.680003333795589, 'random_strength': 3.252168976887568, 'bagging_temperature': 0.4242189672129334, 'border_count': 80}. Best is trial 0 with value: 1.2194420886556452.
[I 2026-05-10 23:10:27,514] Trial 2 finished with value: 1.242251475742872 and parameters: {'learning_rate': 0.0014773610597249873, 'depth': 6, 'l2_leaf_reg': 6.188910129669437, 'random_strength': 1.9816836543125196, 'bagging_temperature': 0.359294780038545, 'border_count': 123}. Best is trial 0 with value: 1.2194420886556452.
[I 2026-05-10 23

KeyboardInterrupt: 